# ArUco-basierte Treffererkennung und Spielumgebung

Dieses Notebook realisiert die markerbasierte Positionsbestimmung der Fahrzeuge sowie die softwareseitige Trefferlogik.

## Funktionen
- Marker-Erkennung
- Positions- und Richtungsbestimmung
- Hitbox-Berechnung
- Spielzustandsverwaltung
- Visualisierung mit PyGame


In [2]:
import time
import cv2
import numpy as np
from cv2 import aruco
import pygame
from dataclasses import dataclass, field
from typing import Optional, Tuple

# =============================
# ArUco Konfiguration
# =============================
CALIB_FILE = "camera_calib_neu.npz"

ID_SHOOTER = 0
ID_TARGET = 1

MARKER_LENGTH_M = 0.056
HIT_RADIUS_M = 0.11

CAM_INDEX = 0

SHOW_CV_WINDOW = True
SHOT_OVERLAY_S = 1.5

# Robustheitsparameter
HIT_RADIUS_SCALE = 1.08   # kleine Toleranz gegen Jitter
FRONT_IS_MINUS_Y = True   # True/False je nach Marker-Front

# =============================
# Pygame UI Konfiguration
# =============================
WIDTH, HEIGHT = 1280, 720
FPS = 60

BLUE = (59, 130, 246)
RED = (239, 68, 68)
BG = (11, 15, 25)
WHITE = (229, 231, 235)
GRAY = (150, 160, 170)

PH_HOME = "HOME"
PH_SETTINGS = "SETTINGS"
PH_COUNTDOWN = "COUNTDOWN"
PH_INGAME = "INGAME"
PH_GAMEOVER = "GAMEOVER"

XBOX_LB_BTN = 4
XBOX_RB_BTN = 5

# =============================
# Helper
# =============================
def put_text(frame, text, org, scale=0.7):
    cv2.putText(frame, text, org, cv2.FONT_HERSHEY_SIMPLEX, scale, (0, 0, 0), 3, cv2.LINE_AA)
    cv2.putText(frame, text, org, cv2.FONT_HERSHEY_SIMPLEX, scale, (255, 255, 255), 1, cv2.LINE_AA)

def normalize(v):
    v = np.asarray(v, dtype=np.float64).reshape(-1)
    n = np.linalg.norm(v)
    if n < 1e-9:
        return np.array([1.0, 0.0], dtype=np.float64)
    return v / n

def ray_circle_hit_2d(o, fwd, p, hit_radius):
    """
    2D-Ray-vs-Circle:
      Ray: x(t) = o + t*fwd, t >= 0
      Circle: ||x - p|| = hit_radius

    Rückgabe:
      hit, d_center, angle_deg, theta_deg, impact_point, closest_point, miss_dist
    """
    o = np.asarray(o, dtype=np.float64)
    fwd = normalize(fwd)
    p = np.asarray(p, dtype=np.float64)

    to_target = p - o
    d_center = float(np.linalg.norm(to_target))

    if d_center < 1e-9:
        return True, 0.0, 0.0, 90.0, p.copy(), p.copy(), 0.0

    to_unit = to_target / d_center
    cosang = float(np.clip(np.dot(fwd, to_unit), -1.0, 1.0))
    angle_deg = float(np.degrees(np.arccos(cosang)))
    theta_deg = float(np.degrees(np.arctan(hit_radius / d_center)))

    t_closest = float(np.dot(to_target, fwd))

    if t_closest < 0:
        closest_point = o.copy()
        miss_dist = float(np.linalg.norm(p - closest_point))
        return False, d_center, angle_deg, theta_deg, None, closest_point, miss_dist

    closest_point = o + t_closest * fwd
    miss_dist = float(np.linalg.norm(p - closest_point))

    if miss_dist > hit_radius:
        return False, d_center, angle_deg, theta_deg, None, closest_point, miss_dist

    thc = float(np.sqrt(max(hit_radius**2 - miss_dist**2, 0.0)))
    t_entry = t_closest - thc
    if t_entry < 0:
        t_entry = t_closest + thc
        if t_entry < 0:
            return False, d_center, angle_deg, theta_deg, None, closest_point, miss_dist

    impact_point = o + t_entry * fwd
    return True, d_center, angle_deg, theta_deg, impact_point, closest_point, miss_dist

def stable_marker_direction_from_corners(corners, prev_dir=None):
    """
    Stabile 2D-Richtung direkt aus den Marker-Ecken im Bild.
    OpenCV-ArUco-Eckreihenfolge: tl, tr, br, bl
    """
    pts = corners.reshape(4, 2).astype(np.float64)
    tl, tr, br, bl = pts

    top_mid = 0.5 * (tl + tr)
    bottom_mid = 0.5 * (bl + br)

    if FRONT_IS_MINUS_Y:
        raw_dir = top_mid - bottom_mid
    else:
        raw_dir = bottom_mid - top_mid

    raw_dir = normalize(raw_dir)

    # 180°-Flip vermeiden
    if prev_dir is not None and np.dot(raw_dir, prev_dir) < 0:
        raw_dir = -raw_dir

    center = 0.25 * (tl + tr + br + bl)
    return center, raw_dir

def hit_radius_pixels(corners):
    pts = corners.reshape(4, 2).astype(np.float64)
    tl, tr, br, bl = pts

    w1 = np.linalg.norm(tr - tl)
    w2 = np.linalg.norm(br - bl)
    marker_px = 0.5 * (w1 + w2)

    px_per_m = marker_px / MARKER_LENGTH_M
    return HIT_RADIUS_M * HIT_RADIUS_SCALE * px_per_m

# =============================
# Game State
# =============================
@dataclass
class Team:
    name: str
    color: Tuple[int, int, int]
    lives: int

@dataclass
class Config:
    duration_s: int = 5 * 60
    lives: int = 5
    fire_rate_hz: float = 2.0

@dataclass
class Game:
    phase: str = PH_HOME
    cfg: Config = field(default_factory=Config)
    team1: Team = field(default_factory=lambda: Team("Team A", BLUE, 5))
    team2: Team = field(default_factory=lambda: Team("Team B", RED, 5))

    can_move: bool = False
    remaining_s: int = 0
    countdown: float = 0.0
    winner: Optional[int] = None

G = Game()

def fmt_time(s: int) -> str:
    m = s // 60
    r = s % 60
    return f"{m:02d}:{r:02d}"

def register_hit(target_id: int):
    if G.phase != PH_INGAME:
        return
    team = G.team1 if target_id == 1 else G.team2
    team.lives = max(0, team.lives - 1)
    if team.lives == 0:
        end_game(2 if target_id == 1 else 1)

def end_game(winner: Optional[int]):
    G.phase = PH_GAMEOVER
    G.can_move = False
    G.winner = winner

def apply_settings(duration_s: int, lives: int, fire_rate: float, name1: str, name2: str):
    G.cfg.duration_s = duration_s
    G.cfg.lives = lives
    G.cfg.fire_rate_hz = fire_rate

    G.team1.name = name1.strip() or "Team A"
    G.team2.name = name2.strip() or "Team B"

    G.team1.lives = lives
    G.team2.lives = lives

# =============================
# UI Widgets
# =============================
class Button:
    def __init__(self, rect, text, on_click, color=(60, 120, 200)):
        self.rect = pygame.Rect(rect)
        self.text = text
        self.on_click = on_click
        self.color = color

    def draw(self, screen, font):
        pygame.draw.rect(screen, self.color, self.rect, border_radius=12)
        lbl = font.render(self.text, True, (255, 255, 255))
        screen.blit(lbl, lbl.get_rect(center=self.rect.center))

    def handle(self, ev):
        if ev.type == pygame.MOUSEBUTTONDOWN and ev.button == 1:
            if self.rect.collidepoint(ev.pos):
                self.on_click()

class CycleOption:
    def __init__(self, label, options, value_index=0):
        self.label = label
        self.options = options
        self.idx = value_index

    @property
    def value(self):
        return self.options[self.idx][1]

    @property
    def display(self):
        return self.options[self.idx][0]

    def cycle(self, delta):
        self.idx = (self.idx + delta) % len(self.options)

class TextField:
    def __init__(self, rect, text=""):
        self.rect = pygame.Rect(rect)
        self.text = text
        self.active = False

    def handle(self, ev):
        if ev.type == pygame.MOUSEBUTTONDOWN and ev.button == 1:
            self.active = self.rect.collidepoint(ev.pos)

        if self.active and ev.type == pygame.KEYDOWN:
            if ev.key == pygame.K_BACKSPACE:
                self.text = self.text[:-1]
            elif ev.key == pygame.K_RETURN:
                self.active = False
            else:
                if len(ev.unicode) == 1 and len(self.text) < 18:
                    self.text += ev.unicode

    def draw(self, screen, font):
        col = (255, 255, 255) if self.active else (200, 200, 200)
        pygame.draw.rect(screen, (20, 26, 40), self.rect, border_radius=10)
        pygame.draw.rect(screen, col, self.rect, 2, border_radius=10)
        lbl = font.render(self.text, True, WHITE)
        screen.blit(lbl, (self.rect.x + 10, self.rect.y + 8))

def draw_panel(screen, rect):
    surf = pygame.Surface((rect.width, rect.height), pygame.SRCALPHA)
    surf.fill((255, 255, 255, 18))
    pygame.draw.rect(surf, (255, 255, 255, 40), surf.get_rect(), 1, border_radius=18)
    screen.blit(surf, rect.topleft)

# =============================
# MAIN
# =============================
def main():
    aruco_dict = aruco.getPredefinedDictionary(aruco.DICT_4X4_50)
    params = aruco.DetectorParameters()
    detector = aruco.ArucoDetector(aruco_dict, params)

    cap = cv2.VideoCapture(CAM_INDEX, cv2.CAP_DSHOW)
    if not cap.isOpened():
        raise RuntimeError(f"Kamera {CAM_INDEX} konnte nicht geöffnet werden.")

    cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)
    cap.set(cv2.CAP_PROP_FPS, 30)
    cap.set(cv2.CAP_PROP_BUFFERSIZE, 1)

    pygame.init()
    pygame.display.set_caption("Panzer-Spiel UI")
    screen = pygame.display.set_mode((WIDTH, HEIGHT))
    clock = pygame.time.Clock()

    pygame.mixer.init()

    shot_sound = pygame.mixer.Sound("shot.wav")
    hit_sound = pygame.mixer.Sound("hit.wav")

    shot_sound.set_volume(0.2)
    hit_sound.set_volume(0.5)
    
    font_big = pygame.font.SysFont("Segoe UI", 64, bold=True)
    font_mid = pygame.font.SysFont("Segoe UI", 32, bold=True)
    font_sm = pygame.font.SysFont("Segoe UI", 22)

    pygame.joystick.init()
    joy = None
    if pygame.joystick.get_count() > 0:
        joy = pygame.joystick.Joystick(0)
        joy.init()
        print("Gamepad verbunden:", joy.get_name())
    else:
        print("Kein Gamepad gefunden – Shoot per SPACE bleibt aktiv.")

    opt_time = CycleOption("Spielzeit", [("5 min", 300), ("10 min", 600), ("15 min", 900), ("20 min", 1200)], 0)
    opt_lives = CycleOption("Leben", [("1", 1), ("3", 3), ("5", 5), ("7", 7), ("10", 10)], 2)
    opt_rate = CycleOption(
    "Schussrate",
    [
        
        ("1 / 2s", 0.5),
        ("1 / 3s", 1.0/3.0),
        ("1 / 4s", 0.25),
        ("1 / 5s", 0.2),
        ("1 / 6s", .16),
    ],
    0
)

    tf1 = TextField((WIDTH // 2 - 220, 320, 440, 44), "Team A")
    tf2 = TextField((WIDTH // 2 - 220, 380, 440, 44), "Team B")

    apply_settings(opt_time.value, opt_lives.value, opt_rate.value, tf1.text, tf2.text)

    buttons = []
    last_second = time.time()

    last_shot_t = 0.0
    last_shot_result = None
    last_shot_info = ""
    last_shot_overlay_t = 0.0

    prev_dirs = {}

    running = True
    while running:
        dt = clock.tick(FPS) / 1000.0
        now = time.time()

        poses2d = {}
        target_radius_px = None
        ids_flat = None
        corners_current = None

        ret, frame = cap.read()
        if ret:
            corners, ids, _ = detector.detectMarkers(frame)

            if ids is not None:
                ids_flat = ids.flatten().astype(int)
                corners_current = corners

                # Marker-Umrandung + ID
                aruco.drawDetectedMarkers(frame, corners, ids)

                for i, mid in enumerate(ids_flat):
                    mid = int(mid)

                    center_px, fwd_px = stable_marker_direction_from_corners(
                        corners[i],
                        prev_dir=prev_dirs.get(mid)
                    )

                    prev_dirs[mid] = fwd_px
                    poses2d[mid] = (center_px, fwd_px)

                if ID_SHOOTER in poses2d and ID_TARGET in poses2d:
                    o_xy, fwd_xy = poses2d[ID_SHOOTER]
                    p_xy, _ = poses2d[ID_TARGET]

                    target_idx = list(ids_flat).index(ID_TARGET)
                    target_radius_px = hit_radius_pixels(corners[target_idx])

                    hit_preview, d_preview, ang_preview, theta_preview, impact_preview, closest_preview, miss_preview = ray_circle_hit_2d(
                        o_xy, fwd_xy, p_xy, target_radius_px
                    )

                    # Hitbox direkt in 2D zeichnen
                    cv2.circle(
                        frame,
                        (int(p_xy[0]), int(p_xy[1])),
                        int(target_radius_px),
                        (0, 255, 0),
                        2,
                        cv2.LINE_AA
                    )

                    # Schusslinie als feste 2D-Gerade bis zum Bildrand
                    line_len_px = 1200
                    x0, y0 = int(o_xy[0]), int(o_xy[1])
                    x1 = int(o_xy[0] + fwd_xy[0] * line_len_px)
                    y1 = int(o_xy[1] + fwd_xy[1] * line_len_px)

                    h, w = frame.shape[:2]
                    ok, pt1, pt2 = cv2.clipLine((0, 0, w, h), (x0, y0), (x1, y1))
                    if ok:
                        cv2.line(
                            frame,
                            pt1,
                            pt2,
                            (0, 255, 255) if hit_preview else (0, 180, 255),
                            2,
                            cv2.LINE_AA
                        )

                    put_text(
                        frame,
                        f"preview: {'HIT' if hit_preview else 'MISS'} | d={d_preview:.2f}px angle={ang_preview:.1f}deg tol={theta_preview:.1f}deg",
                        (10, 35),
                        0.75
                    )
                    if not hit_preview:
                        put_text(frame, f"miss_dist={miss_preview:.1f}px", (10, 70), 0.75)

            if last_shot_result is not None and (now - last_shot_overlay_t) < SHOT_OVERLAY_S:
                status = "HIT" if last_shot_result else "NO HIT"
                put_text(frame, f"SHOT: {status} | {last_shot_info}", (10, 105), 0.8)

            if SHOW_CV_WINDOW:
                cv2.imshow("ArUco Live", frame)
                cv2.waitKey(1)

        shoot_edge = False
        for ev in pygame.event.get():
            if ev.type == pygame.QUIT:
                running = False

            if ev.type == pygame.KEYDOWN:
                if ev.key == pygame.K_ESCAPE:
                    running = False
                if ev.key == pygame.K_F11:
                    pygame.display.toggle_fullscreen()
                if ev.key == pygame.K_SPACE:
                    shoot_edge = True

            if G.phase == PH_SETTINGS:
                tf1.handle(ev)
                tf2.handle(ev)

            for b in buttons:
                b.handle(ev)

            if ev.type == pygame.JOYBUTTONDOWN:
                if ev.button in (XBOX_LB_BTN, XBOX_RB_BTN):
                    shoot_edge = True

            if G.phase == PH_SETTINGS and ev.type == pygame.MOUSEBUTTONDOWN and ev.button == 1:
                mx, my = ev.pos

                def arrow_hit(y):
                    left = pygame.Rect(WIDTH // 2 - 260, y, 30, 30)
                    right = pygame.Rect(WIDTH // 2 + 230, y, 30, 30)
                    return left.collidepoint(mx, my), right.collidepoint(mx, my)

                l, r = arrow_hit(200)
                if l:
                    opt_time.cycle(-1)
                if r:
                    opt_time.cycle(+1)

                l, r = arrow_hit(240)
                if l:
                    opt_lives.cycle(-1)
                if r:
                    opt_lives.cycle(+1)

                l, r = arrow_hit(280)
                if l:
                    opt_rate.cycle(-1)
                if r:
                    opt_rate.cycle(+1)

        shoot_held = False
        if joy is not None:
            try:
                shoot_held = bool(joy.get_button(XBOX_LB_BTN) or joy.get_button(XBOX_RB_BTN))
            except pygame.error:
                shoot_held = False

        shoot_pressed = shoot_edge or shoot_held

        if G.phase == PH_COUNTDOWN:
            G.countdown -= dt
            if G.countdown <= 0:
                G.phase = PH_INGAME
                G.can_move = True
                G.remaining_s = G.cfg.duration_s
                last_second = now

        if G.phase == PH_INGAME:
            if now - last_second >= 1.0:
                last_second += 1.0
                G.remaining_s -= 1
                if G.remaining_s <= 0:
                    if G.team1.lives == G.team2.lives:
                        end_game(None)
                    else:
                        end_game(1 if G.team1.lives > G.team2.lives else 2)

        if shoot_pressed and G.phase == PH_INGAME:
            fire_rate = max(0.1, float(G.cfg.fire_rate_hz))
            min_interval = 1.0 / fire_rate

            if (now - last_shot_t) >= min_interval:
                last_shot_t = now
                
                shot_sound.play()
                
                if ID_SHOOTER in poses2d and ID_TARGET in poses2d and ids_flat is not None and corners_current is not None:
                    o_xy, fwd_xy = poses2d[ID_SHOOTER]
                    p_xy, _ = poses2d[ID_TARGET]

                    target_idx = list(ids_flat).index(ID_TARGET)
                    target_radius_px = hit_radius_pixels(corners_current[target_idx])

                    hit, d, ang, theta, impact_point, closest_point, miss_dist = ray_circle_hit_2d(
                        o_xy, fwd_xy, p_xy, target_radius_px
                    )

                    last_shot_result = hit
                    if hit:
                        hit_sound.play()
                        last_shot_info = f"d={d:.2f}px angle={ang:.1f}deg tol={theta:.1f}deg"
                        register_hit(ID_TARGET)
                    else:
                        last_shot_info = f"d={d:.2f}px angle={ang:.1f}deg tol={theta:.1f}deg miss={miss_dist:.1f}px"

                    last_shot_overlay_t = now
                else:
                    last_shot_result = False
                    last_shot_info = "missing shooter/target marker"
                    last_shot_overlay_t = now

        screen.fill(BG)
        buttons = []

        if G.phase == PH_HOME:
            title = font_big.render("Panzer-Spiel", True, WHITE)
            screen.blit(title, title.get_rect(center=(WIDTH // 2, 160)))

            def go_settings():
                G.phase = PH_SETTINGS

            buttons.append(Button((WIDTH // 2 - 140, 260, 280, 54), "Zum Spielmenü", go_settings, color=(59, 130, 246)))

        elif G.phase == PH_SETTINGS:
            draw_panel(screen, pygame.Rect(WIDTH // 2 - 320, 120, 640, 520))
            screen.blit(font_mid.render("Voreinstellungen", True, WHITE), (WIDTH // 2 - 190, 140))

            def draw_cycle(y, opt: CycleOption):
                screen.blit(font_sm.render(f"{opt.label}:", True, GRAY), (WIDTH // 2 - 220, y))
                screen.blit(font_sm.render(opt.display, True, WHITE), (WIDTH // 2 - 40, y))
                pygame.draw.polygon(screen, WHITE, [(WIDTH // 2 - 245, y + 8), (WIDTH // 2 - 235, y + 15), (WIDTH // 2 - 245, y + 22)])
                pygame.draw.polygon(screen, WHITE, [(WIDTH // 2 + 245, y + 8), (WIDTH // 2 + 255, y + 15), (WIDTH // 2 + 245, y + 22)])

            draw_cycle(200, opt_time)
            draw_cycle(240, opt_lives)
            draw_cycle(280, opt_rate)

            screen.blit(font_sm.render("Team 1:", True, GRAY), (WIDTH // 2 - 300, 330))
            screen.blit(font_sm.render("Team 2:", True, GRAY), (WIDTH // 2 - 300, 390))
            tf1.draw(screen, font_sm)
            tf2.draw(screen, font_sm)

            def apply():
                apply_settings(opt_time.value, opt_lives.value, opt_rate.value, tf1.text, tf2.text)

            def start():
                apply()
                G.phase = PH_COUNTDOWN
                G.can_move = False
                G.winner = None
                G.countdown = 3.0
                G.team1.lives = G.cfg.lives
                G.team2.lives = G.cfg.lives

            def back():
                G.phase = PH_HOME

            buttons.append(Button((WIDTH // 2 - 280, 470, 180, 50), "Übernehmen", apply, color=(59, 130, 246)))
            buttons.append(Button((WIDTH // 2 - 90, 470, 180, 50), "Spiel starten", start, color=(34, 197, 94)))
            buttons.append(Button((WIDTH // 2 + 100, 470, 180, 50), "Zurück", back, color=(107, 114, 128)))

        elif G.phase == PH_COUNTDOWN:
            screen.blit(font_mid.render("Spiel startet in...", True, WHITE), (WIDTH // 2 - 150, 180))
            n = max(0, int(G.countdown) + 1)
            screen.blit(font_big.render(str(n), True, WHITE), (WIDTH // 2 - 20, 260))

        elif G.phase == PH_INGAME:
            screen.blit(font_big.render(fmt_time(G.remaining_s), True, WHITE), (WIDTH // 2 - 110, 40))

            l1 = font_mid.render(f"{G.team1.name}:  ❤ {G.team1.lives}", True, G.team1.color)
            l2 = font_mid.render(f"{G.team2.name}:  ❤ {G.team2.lives}", True, G.team2.color)
            screen.blit(l1, (80, 140))
            screen.blit(l2, (80, 190))

            hint = font_sm.render(
                f"LB/RB oder SPACE=shoot | fire_rate={G.cfg.fire_rate_hz}/s | can_move={G.can_move} | F11=Fullscreen",
                True, GRAY
            )
            screen.blit(hint, (80, HEIGHT - 40))

        elif G.phase == PH_GAMEOVER:
            if G.winner is None:
                msg = "Unentschieden"
                col = WHITE
            else:
                team = G.team1 if G.winner == 1 else G.team2
                msg = f"{team.name} Verliert!"
                col = team.color

            screen.blit(font_big.render("GAME OVER", True, WHITE), (WIDTH // 2 - 180, 160))
            screen.blit(font_mid.render(msg, True, col), (WIDTH // 2 - 220, 260))

            def back_to_menu():
                G.phase = PH_SETTINGS
                G.can_move = False
                G.winner = None
                G.team1.lives = G.cfg.lives
                G.team2.lives = G.cfg.lives

            buttons.append(Button((WIDTH // 2 - 120, 360, 240, 54), "Zurück ins Menü", back_to_menu, color=(59, 130, 246)))

        for b in buttons:
            b.draw(screen, font_sm)

        pygame.display.flip()

    cap.release()
    if SHOW_CV_WINDOW:
        cv2.destroyAllWindows()
    pygame.quit()

if __name__ == "__main__":
    main()

Gamepad verbunden: Xbox Series X Controller
